# Multimodal Sentiment Analysis Using Text, Audio, and Video

## Introduction

Multimodal sentiment analysis predicts sentiment or emotion by combining more than one type of input. In this project, the three intended modalities are:

1. **Text**: utterance/caption/transcript.
2. **Audio**: tone, pitch, speaking energy, and rhythm.
3. **Video**: facial expressions, motion, and visual cues.

This notebook now supports the **MELD raw dataset**. MELD provides text utterances, `.mp4` video clips, and sentiment/emotion labels. The `.mp4` files are used for both audio and video feature extraction.

If `MELD-RAW/MELD.Raw` is present, the notebook automatically uses MELD. If MELD is not present, it falls back to the earlier `LabeledText.xlsx` dataset.


## Problem Statement

The goal is to classify sentiment using multimodal information.

For MELD, each row contains:

- `Utterance`: text spoken by a character.
- `Sentiment`: positive, negative, or neutral.
- `Emotion`: joy, anger, sadness, surprise, fear, disgust, or neutral.
- `Dialogue_ID` and `Utterance_ID`: used to locate the matching `.mp4` file.

The default target in this notebook is **sentiment classification**:

- `negative`
- `neutral`
- `positive`

The same dataset can also support emotion classification by changing the target setting.


## 1. Optional Dependency Installation

Run this cell only if your environment is missing required packages.

For real audio/video analysis, install:

```bash
pip install pandas numpy scikit-learn tensorflow librosa opencv-python matplotlib seaborn openpyxl
```

Audio extraction from `.mp4` may also require **FFmpeg** on your system. On macOS with Homebrew:

```bash
brew install ffmpeg
```

After installing packages, restart the Jupyter kernel and run the notebook again.


In [1]:
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    import sys
    import subprocess

    packages = [
        "pandas",
        "numpy",
        "scikit-learn",
        "tensorflow",
        "librosa",
        "opencv-python",
        "matplotlib",
        "seaborn",
        "openpyxl",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])
    print("Dependencies installed. Restart the kernel before continuing.")
else:
    print("Dependency installation skipped. Set INSTALL_DEPENDENCIES = True only if packages are missing.")


Dependency installation skipped. Set INSTALL_DEPENDENCIES = True only if packages are missing.


## 2. Import Required Libraries

This section imports all required libraries. Optional libraries are handled with `try-except` so the notebook does not crash when a package is missing.

The notebook works in two modes:

1. **Full mode**: uses TensorFlow, scikit-learn, librosa, and OpenCV when installed.
2. **Fallback mode**: uses NumPy-based replacements and zero-vector audio/video features when optional packages are missing.


In [ ]:
import os
import re
import math
import shutil
import hashlib
import warnings
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

try:
    display
except NameError:
    try:
        from IPython.display import display
    except Exception:
        display = print


try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception as error:
    MATPLOTLIB_AVAILABLE = False
    plt = None
    print("matplotlib unavailable. Plots will be skipped.")

try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except Exception as error:
    SEABORN_AVAILABLE = False
    sns = None
    print("seaborn unavailable. Confusion matrix heatmap will use table fallback.")

try:
    from sklearn.model_selection import train_test_split as sklearn_train_test_split
    from sklearn.preprocessing import LabelEncoder as SklearnLabelEncoder
    from sklearn.preprocessing import StandardScaler as SklearnStandardScaler
    from sklearn.feature_extraction.text import TfidfVectorizer as SklearnTfidfVectorizer
    from sklearn.metrics import classification_report as sklearn_classification_report
    from sklearn.metrics import confusion_matrix as sklearn_confusion_matrix
    from sklearn.metrics import precision_recall_fscore_support as sklearn_prfs
    SKLEARN_AVAILABLE = True
except Exception as error:
    SKLEARN_AVAILABLE = False
    print("scikit-learn unavailable. NumPy fallback utilities will be used.")

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout, Input
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.utils import to_categorical
    TF_AVAILABLE = True
except Exception as error:
    TF_AVAILABLE = False
    tf = None
    print("TensorFlow unavailable. NumPy neural network fallback will be used.")

    def to_categorical(y, num_classes):
        y = np.asarray(y, dtype=int)
        return np.eye(num_classes, dtype=np.float32)[y]

try:
    import librosa
    LIBROSA_AVAILABLE = True
except Exception as error:
    LIBROSA_AVAILABLE = False
    librosa = None
    print("librosa unavailable. Real MFCC audio extraction will be skipped.")

try:
    import cv2
    CV2_AVAILABLE = True
except Exception as error:
    CV2_AVAILABLE = False
    cv2 = None
    print("OpenCV unavailable. Real video frame extraction will be skipped.")

try:
    from transformers import AutoTokenizer, TFAutoModel
    TRANSFORMERS_AVAILABLE = True
except Exception as error:
    TRANSFORMERS_AVAILABLE = False
    AutoTokenizer = None
    TFAutoModel = None
    print("transformers unavailable. TF-IDF text features will be used.")


RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

if TF_AVAILABLE:
    tf.random.set_seed(RANDOM_STATE)

if SEABORN_AVAILABLE:
    sns.set_theme(style="whitegrid")

FFMPEG_AVAILABLE = shutil.which("ffmpeg") is not None

print("Library status")
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn available:", SKLEARN_AVAILABLE)
print("TensorFlow available:", TF_AVAILABLE)
print("librosa available:", LIBROSA_AVAILABLE)
print("OpenCV available:", CV2_AVAILABLE)
print("FFmpeg available:", FFMPEG_AVAILABLE)
print("Matplotlib available:", MATPLOTLIB_AVAILABLE)
print("Seaborn available:", SEABORN_AVAILABLE)
print("Transformers available:", TRANSFORMERS_AVAILABLE)


## 3. Project Configuration

MELD contains more than 13,000 utterances and video clips. Extracting MFCC and video-frame features from every clip can take a long time.

For a faster project run, this notebook uses **quick-run mode** by default. It samples a balanced subset from train/dev/test and extracts real audio/video features for that subset when dependencies are installed.

To train on more data, increase the sample limits or set:

```python
USE_QUICK_RUN = False
```


In [ ]:
TARGET_TASK = "sentiment"  # Options: "sentiment" or "emotion"

USE_QUICK_RUN = True
QUICK_SAMPLES_PER_CLASS = {
    "train": 120,
    "dev": 40,
    "test": 50,
}

MAX_TFIDF_FEATURES = 1200

EXTRACT_REAL_AUDIO = True
EXTRACT_REAL_VIDEO = True
USE_FEATURE_CACHE = True
FEATURE_CACHE_DIR = Path(".feature_cache")

N_MFCC = 40
AUDIO_SAMPLE_RATE = 16000
MAX_AUDIO_SECONDS = 3.0

VIDEO_FEATURE_DIM = 128
MAX_VIDEO_FRAMES = 8
VIDEO_FRAME_SIZE = (64, 64)

print("Target task:", TARGET_TASK)
print("Quick-run mode:", USE_QUICK_RUN)
print("Feature cache:", USE_FEATURE_CACHE)


## 4. Load MELD Dataset

The notebook searches for MELD in this location:

```text
MELD-RAW/MELD.Raw/
```

Expected MELD files:

```text
MELD-RAW/MELD.Raw/train/train_sent_emo.csv
MELD-RAW/MELD.Raw/dev_sent_emo.csv
MELD-RAW/MELD.Raw/test_sent_emo.csv
MELD-RAW/MELD.Raw/train/train_splits/
MELD-RAW/MELD.Raw/dev/dev_splits_complete/
MELD-RAW/MELD.Raw/test/output_repeated_splits_test/
```

The `.mp4` filename is created using:

```text
dia{Dialogue_ID}_utt{Utterance_ID}.mp4
```


In [ ]:
def find_meld_root():
    candidates = [
        Path("MELD-RAW/MELD.Raw"),
        Path("MELD.Raw"),
        Path("MELD-RAW"),
    ]

    for candidate in candidates:
        train_csv = candidate / "train" / "train_sent_emo.csv"
        dev_csv = candidate / "dev_sent_emo.csv"
        test_csv = candidate / "test_sent_emo.csv"
        if train_csv.exists() and dev_csv.exists() and test_csv.exists():
            return candidate

    return None


MELD_ROOT = find_meld_root()
print("MELD root:", MELD_ROOT.resolve() if MELD_ROOT else "Not found")


In [ ]:
def load_meld_split(meld_root, split):
    if split == "train":
        csv_path = meld_root / "train" / "train_sent_emo.csv"
        media_dir = meld_root / "train" / "train_splits"
    elif split == "dev":
        csv_path = meld_root / "dev_sent_emo.csv"
        media_dir = meld_root / "dev" / "dev_splits_complete"
    elif split == "test":
        csv_path = meld_root / "test_sent_emo.csv"
        media_dir = meld_root / "test" / "output_repeated_splits_test"
    else:
        raise ValueError(f"Unknown MELD split: {split}")

    split_df = pd.read_csv(csv_path)
    split_df["split"] = split
    split_df["file_id"] = split_df.apply(
        lambda row: f"dia{int(row['Dialogue_ID'])}_utt{int(row['Utterance_ID'])}",
        axis=1,
    )
    split_df["video_path"] = split_df["file_id"].apply(lambda file_id: str(media_dir / f"{file_id}.mp4"))
    split_df["audio_path"] = split_df["video_path"]
    return split_df


def load_meld_dataset(meld_root):
    parts = [load_meld_split(meld_root, split) for split in ["train", "dev", "test"]]
    meld_df = pd.concat(parts, ignore_index=True)

    if TARGET_TASK == "emotion":
        label_column = "Emotion"
    else:
        label_column = "Sentiment"

    standardized = pd.DataFrame(index=meld_df.index)
    standardized["dataset"] = "MELD"
    standardized["split"] = meld_df["split"]
    standardized["file_id"] = meld_df["file_id"]
    standardized["text"] = meld_df["Utterance"].astype(str)
    standardized["speaker"] = meld_df["Speaker"].astype(str)
    standardized["emotion"] = meld_df["Emotion"].astype(str).str.lower()
    standardized["sentiment"] = meld_df["Sentiment"].astype(str).str.lower()
    standardized["label"] = meld_df[label_column].astype(str).str.lower()
    standardized["audio_path"] = meld_df["audio_path"]
    standardized["video_path"] = meld_df["video_path"]
    return standardized


if MELD_ROOT is not None:
    df = load_meld_dataset(MELD_ROOT)
    DATASET_USED = "MELD"
else:
    DATASET_USED = None

print("Dataset used:", DATASET_USED)
if DATASET_USED == "MELD":
    print("Full MELD dataframe shape:", df.shape)
    print("Split counts:")
    display(df["split"].value_counts().rename_axis("split").reset_index(name="count"))
    print("Label distribution:")
    display(df["label"].value_counts().rename_axis("label").reset_index(name="count"))
    display(df.head())


## 5. Fallback Dataset: `LabeledText.xlsx`

If MELD is not available, the notebook falls back to `LabeledText.xlsx`.

This fallback is text-only. It keeps the notebook runnable, but real audio/video analysis requires MELD or another dataset with media files.


In [ ]:
def load_labeled_text_dataset():
    dataset_path = Path("LabeledText.xlsx")
    if not dataset_path.exists():
        raise FileNotFoundError(
            "Neither MELD-RAW/MELD.Raw nor LabeledText.xlsx was found. "
            "Add one dataset and run again."
        )

    raw_df = pd.read_excel(dataset_path, sheet_name="final label")
    fallback_df = pd.DataFrame(index=raw_df.index)
    fallback_df["dataset"] = "LabeledText"
    fallback_df["split"] = "unsplit"
    fallback_df["file_id"] = raw_df["File Name"].astype(str)
    fallback_df["text"] = raw_df["Caption"].astype(str)
    fallback_df["speaker"] = ""
    fallback_df["emotion"] = ""
    fallback_df["sentiment"] = raw_df["LABEL"].astype(str).str.lower().str.strip()
    fallback_df["label"] = fallback_df["sentiment"]
    fallback_df["audio_path"] = ""
    fallback_df["video_path"] = ""
    return fallback_df.dropna(subset=["text", "label"]).reset_index(drop=True)


if DATASET_USED is None:
    df = load_labeled_text_dataset()
    DATASET_USED = "LabeledText"
    print("Dataset used:", DATASET_USED)
    print("Dataset shape:", df.shape)
    print("Label distribution:")
    display(df["label"].value_counts().rename_axis("label").reset_index(name="count"))
    display(df.head())


## 6. Quick-Run Sampling

In quick-run mode, the notebook samples a balanced number of examples per class from each split.

This makes real audio/video feature extraction practical for a laptop. You can turn quick-run off later for larger experiments.


In [ ]:
def sample_split_by_class(input_df, split_name, samples_per_class, label_column="label"):
    split_df = input_df[input_df["split"] == split_name].copy()
    if split_df.empty:
        return split_df

    sampled_parts = []
    for label, group in split_df.groupby(label_column):
        n = min(samples_per_class, len(group))
        sampled_parts.append(group.sample(n=n, random_state=RANDOM_STATE))

    return pd.concat(sampled_parts, ignore_index=True)


if DATASET_USED == "MELD" and USE_QUICK_RUN:
    sampled_parts = [
        sample_split_by_class(df, "train", QUICK_SAMPLES_PER_CLASS["train"]),
        sample_split_by_class(df, "dev", QUICK_SAMPLES_PER_CLASS["dev"]),
        sample_split_by_class(df, "test", QUICK_SAMPLES_PER_CLASS["test"]),
    ]
    df = pd.concat(sampled_parts, ignore_index=True)
elif DATASET_USED == "LabeledText" and USE_QUICK_RUN:
    df = (
        df.groupby("label", group_keys=False)
        .apply(lambda group: group.sample(n=min(600, len(group)), random_state=RANDOM_STATE))
        .reset_index(drop=True)
    )

df = df.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

print("Working dataset:", DATASET_USED)
print("Working dataframe shape:", df.shape)
print("Split counts:")
display(df["split"].value_counts().rename_axis("split").reset_index(name="count"))
print("Label distribution:")
display(df["label"].value_counts().rename_axis("label").reset_index(name="count"))

existing_media = df["video_path"].apply(lambda path: Path(str(path)).is_file()).sum()
print("Existing video/audio media files found:", int(existing_media))


## 7. Text Preprocessing

The text is cleaned before feature extraction.

The cleaning steps are:

1. Lowercase text.
2. Remove URLs and user mentions.
3. Convert hashtags into normal words.
4. Remove special characters.
5. Remove extra spaces.


In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = (
        text.replace("\x92", "'")
        .replace("\u0092", "'")
        .replace("\u2018", "'")
        .replace("\u2019", "'")
        .replace("\u2014", " ")
        .replace("\x97", " ")
    )
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = text.replace("#", " ")
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df["clean_text"] = df["text"].apply(clean_text)
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)

print("Dataset shape after text cleaning:", df.shape)
display(df[["text", "clean_text", "label", "split", "video_path"]].head())


## 8. Text Feature Extraction

The text modality is converted into numerical features.

This notebook uses **TF-IDF** by default. TF-IDF gives higher weight to words that are important in a document but not too common across all documents.

BERT support is kept as an optional upgrade, but TF-IDF is faster and easier for a beginner-level project.


In [ ]:
class SimpleTfidfVectorizer:
    def __init__(self, max_features=1200, min_df=1):
        self.max_features = max_features
        self.min_df = min_df
        self.vocabulary_ = {}
        self.idf_ = None

    def _tokenize(self, text):
        return re.findall(r"[a-z0-9']+", str(text).lower())

    def fit(self, texts):
        document_frequency = Counter()
        term_frequency = Counter()
        n_documents = len(texts)

        for text in texts:
            tokens = self._tokenize(text)
            term_frequency.update(tokens)
            document_frequency.update(set(tokens))

        valid_terms = [
            term for term, df_count in document_frequency.items()
            if df_count >= self.min_df
        ]
        valid_terms.sort(key=lambda term: (-term_frequency[term], term))
        selected_terms = valid_terms[: self.max_features]

        self.vocabulary_ = {term: idx for idx, term in enumerate(selected_terms)}
        self.idf_ = np.zeros(len(self.vocabulary_), dtype=np.float32)

        for term, idx in self.vocabulary_.items():
            df_count = document_frequency[term]
            self.idf_[idx] = math.log((1 + n_documents) / (1 + df_count)) + 1

        return self

    def transform(self, texts):
        matrix = np.zeros((len(texts), len(self.vocabulary_)), dtype=np.float32)
        for row_index, text in enumerate(texts):
            tokens = self._tokenize(text)
            if not tokens:
                continue
            counts = Counter(tokens)
            total_tokens = sum(counts.values())
            for token, count in counts.items():
                col_index = self.vocabulary_.get(token)
                if col_index is not None:
                    matrix[row_index, col_index] = (count / total_tokens) * self.idf_[col_index]

        norms = np.linalg.norm(matrix, axis=1, keepdims=True)
        return (matrix / np.maximum(norms, 1e-12)).astype(np.float32)

    def fit_transform(self, texts):
        return self.fit(texts).transform(texts)


print("TF-IDF vectorizer ready.")


## 9. Audio Feature Extraction Using MFCC

Audio features are extracted from the `.mp4` clips using `librosa`.

The notebook extracts:

- 40 MFCC coefficients.
- Mean MFCC value across time frames.

MFCC features are useful because they summarize speech frequency patterns related to tone, pitch, and emotion.

If `librosa` or FFmpeg support is unavailable, the notebook returns zero-vector audio features and prints a clear message.


In [ ]:
def path_is_file(path_value):
    if path_value is None or pd.isna(path_value):
        return False
    path_text = str(path_value).strip()
    return bool(path_text) and Path(path_text).is_file()


def extract_audio_features(audio_path, n_mfcc=N_MFCC, sr=AUDIO_SAMPLE_RATE, duration=MAX_AUDIO_SECONDS):
    if EXTRACT_REAL_AUDIO and LIBROSA_AVAILABLE and path_is_file(audio_path):
        try:
            signal, sample_rate = librosa.load(str(audio_path), sr=sr, mono=True, duration=duration)
            if signal.size > 0:
                mfcc = librosa.feature.mfcc(y=signal, sr=sample_rate, n_mfcc=n_mfcc)
                return np.mean(mfcc, axis=1).astype(np.float32), True
        except Exception:
            return np.zeros(n_mfcc, dtype=np.float32), False

    return np.zeros(n_mfcc, dtype=np.float32), False


print("Audio extractor ready.")
if DATASET_USED == "MELD" and not LIBROSA_AVAILABLE:
    print("Install librosa and FFmpeg to extract real MFCC audio features from MELD .mp4 files.")


## 10. Video Feature Extraction Using OpenCV

Video features are extracted from MELD `.mp4` clips using OpenCV.

For each video:

1. Read a limited number of frames.
2. Resize frames.
3. Convert frames to grayscale.
4. Downsample each frame to a compact vector.
5. Average frame vectors into one fixed-size representation.

This is a beginner-friendly CNN-style visual representation. In an advanced version, you can replace this with VGG16, ResNet, MobileNet, or a face-emotion CNN.


In [ ]:
def extract_video_features(video_path, feature_dim=VIDEO_FEATURE_DIM, frame_size=VIDEO_FRAME_SIZE, max_frames=MAX_VIDEO_FRAMES):
    if EXTRACT_REAL_VIDEO and CV2_AVAILABLE and path_is_file(video_path):
        try:
            cap = cv2.VideoCapture(str(video_path))
            frame_embeddings = []

            while len(frame_embeddings) < max_frames:
                success, frame = cap.read()
                if not success:
                    break

                frame = cv2.resize(frame, frame_size)
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                compact_frame = cv2.resize(gray, (16, 8)).flatten() / 255.0
                frame_embeddings.append(compact_frame)

            cap.release()

            if frame_embeddings:
                return np.mean(frame_embeddings, axis=0).astype(np.float32), True
        except Exception:
            return np.zeros(feature_dim, dtype=np.float32), False

    return np.zeros(feature_dim, dtype=np.float32), False


print("Video extractor ready.")
if DATASET_USED == "MELD" and not CV2_AVAILABLE:
    print("Install opencv-python to extract real video frame features from MELD .mp4 files.")


## 11. Extract and Cache Audio/Video Features

Feature extraction from video files can take time. Therefore, this notebook caches extracted media features in `.feature_cache/`.

If you install new media libraries later and want to regenerate features, delete the `.feature_cache` folder and rerun the notebook.


In [ ]:
def make_feature_cache_path(feature_name, paths, dimension):
    FEATURE_CACHE_DIR.mkdir(exist_ok=True)
    signature_text = "|".join(map(str, paths[:10])) + f"|n={len(paths)}|dim={dimension}|"
    signature_text += f"librosa={LIBROSA_AVAILABLE}|cv2={CV2_AVAILABLE}|audio={EXTRACT_REAL_AUDIO}|video={EXTRACT_REAL_VIDEO}|"
    signature = hashlib.sha256(signature_text.encode("utf-8")).hexdigest()[:16]
    return FEATURE_CACHE_DIR / f"{DATASET_USED.lower()}_{TARGET_TASK}_{feature_name}_{signature}.npz"


def load_or_extract_media_features(feature_name, paths, extractor, dimension):
    paths = list(paths)
    cache_path = make_feature_cache_path(feature_name, paths, dimension)

    if USE_FEATURE_CACHE and cache_path.exists():
        cached = np.load(cache_path)
        features = cached["features"].astype(np.float32)
        real_count = int(cached["real_count"])
        print(f"Loaded cached {feature_name} features from {cache_path}")
        return features, real_count

    features = []
    real_count = 0

    for index, path in enumerate(paths, start=1):
        vector, used_real_file = extractor(path)
        features.append(vector)
        real_count += int(used_real_file)

        if index == 1 or index % 100 == 0 or index == len(paths):
            print(f"{feature_name}: processed {index}/{len(paths)}")

    features = np.vstack(features).astype(np.float32)

    if USE_FEATURE_CACHE:
        np.savez_compressed(cache_path, features=features, real_count=real_count)
        print(f"Saved cached {feature_name} features to {cache_path}")

    return features, real_count


audio_features, real_audio_count = load_or_extract_media_features(
    "audio_mfcc",
    df["audio_path"].tolist(),
    extract_audio_features,
    N_MFCC,
)

video_features, real_video_count = load_or_extract_media_features(
    "video_frames",
    df["video_path"].tolist(),
    extract_video_features,
    VIDEO_FEATURE_DIM,
)

print("Audio feature matrix shape:", audio_features.shape)
print("Video feature matrix shape:", video_features.shape)
print("Real audio files used:", real_audio_count)
print("Real video files used:", real_video_count)


## 12. Label Encoding

Labels are converted from strings into numeric classes.

For sentiment classification, labels are:

- `negative`
- `neutral`
- `positive`

For emotion classification, labels can include:

- `anger`
- `disgust`
- `fear`
- `joy`
- `neutral`
- `sadness`
- `surprise`


In [ ]:
class SimpleLabelEncoder:
    def fit(self, labels):
        self.classes_ = np.array(sorted(pd.Series(labels).astype(str).unique()))
        self.class_to_index_ = {label: idx for idx, label in enumerate(self.classes_)}
        return self

    def transform(self, labels):
        return np.array([self.class_to_index_[str(label)] for label in labels], dtype=int)

    def fit_transform(self, labels):
        return self.fit(labels).transform(labels)

    def inverse_transform(self, indices):
        return np.array([self.classes_[int(index)] for index in indices])


label_encoder = SklearnLabelEncoder() if SKLEARN_AVAILABLE else SimpleLabelEncoder()
y = label_encoder.fit_transform(df["label"])
class_names = [str(class_name) for class_name in label_encoder.classes_]
num_classes = len(class_names)
y_categorical = to_categorical(y, num_classes=num_classes).astype(np.float32)

print("Classes:", class_names)
print("Number of classes:", num_classes)


## 13. Train/Validation/Test Split

For MELD, the official `train`, `dev`, and `test` splits are used.

For `LabeledText.xlsx`, the notebook creates a train/test split because the file does not provide official splits.


In [ ]:
def fallback_train_test_indices(labels, test_size=0.25, random_state=42):
    rng = np.random.default_rng(random_state)
    train_indices = []
    test_indices = []

    labels = np.asarray(labels)
    for class_id in np.unique(labels):
        class_indices = np.where(labels == class_id)[0]
        rng.shuffle(class_indices)
        n_test = max(1, int(round(len(class_indices) * test_size)))
        test_indices.extend(class_indices[:n_test])
        train_indices.extend(class_indices[n_test:])

    train_indices = np.array(train_indices, dtype=int)
    test_indices = np.array(test_indices, dtype=int)
    rng.shuffle(train_indices)
    rng.shuffle(test_indices)
    return train_indices, test_indices


if DATASET_USED == "MELD":
    train_indices = np.where(df["split"].values == "train")[0]
    val_indices = np.where(df["split"].values == "dev")[0]
    test_indices = np.where(df["split"].values == "test")[0]
else:
    train_indices, test_indices = fallback_train_test_indices(y, test_size=0.25, random_state=RANDOM_STATE)
    train_df_for_val = df.iloc[train_indices]
    train_labels_for_val = y[train_indices]
    relative_train_indices, relative_val_indices = fallback_train_test_indices(
        train_labels_for_val,
        test_size=0.2,
        random_state=RANDOM_STATE,
    )
    val_indices = train_indices[relative_val_indices]
    train_indices = train_indices[relative_train_indices]

print("Train samples:", len(train_indices))
print("Validation samples:", len(val_indices))
print("Test samples:", len(test_indices))


## 14. Fit Text Vectorizer on Training Data

To avoid data leakage, the TF-IDF vectorizer is fitted only on training text and then applied to validation/test text.


In [ ]:
train_texts = df.loc[train_indices, "clean_text"].tolist()
all_texts = df["clean_text"].tolist()

if SKLEARN_AVAILABLE:
    tfidf_vectorizer = SklearnTfidfVectorizer(
        max_features=MAX_TFIDF_FEATURES,
        ngram_range=(1, 2),
        min_df=2,
    )
    tfidf_vectorizer.fit(train_texts)
    text_features = tfidf_vectorizer.transform(all_texts).toarray().astype(np.float32)
    text_feature_method = "sklearn_tfidf"
else:
    tfidf_vectorizer = SimpleTfidfVectorizer(max_features=MAX_TFIDF_FEATURES, min_df=2)
    tfidf_vectorizer.fit(train_texts)
    text_features = tfidf_vectorizer.transform(all_texts)
    text_feature_method = "simple_tfidf"


def transform_text_inputs(texts):
    cleaned_texts = [clean_text(text) for text in texts]
    if text_feature_method == "sklearn_tfidf":
        return tfidf_vectorizer.transform(cleaned_texts).toarray().astype(np.float32)
    return tfidf_vectorizer.transform(cleaned_texts)


print("Text feature method:", text_feature_method)
print("Text feature matrix shape:", text_features.shape)


## 15. Feature Fusion and Normalization

This project uses **early fusion**.

Early fusion means concatenating feature vectors before classification:

```text
fused = [text_features + audio_features + video_features]
```

Then standard scaling is applied. The scaler is fitted only on the training data.


In [ ]:
class SimpleStandardScaler:
    def fit(self, X):
        self.mean_ = np.mean(X, axis=0)
        self.scale_ = np.std(X, axis=0)
        self.scale_[self.scale_ == 0] = 1.0
        return self

    def transform(self, X):
        return ((X - self.mean_) / self.scale_).astype(np.float32)

    def fit_transform(self, X):
        return self.fit(X).transform(X)


fused_features = np.concatenate(
    [text_features, audio_features, video_features],
    axis=1,
).astype(np.float32)

scaler = SklearnStandardScaler() if SKLEARN_AVAILABLE else SimpleStandardScaler()
scaler.fit(fused_features[train_indices])
fused_features_scaled = scaler.transform(fused_features)

X_train = fused_features_scaled[train_indices]
X_val = fused_features_scaled[val_indices]
X_test = fused_features_scaled[test_indices]

y_train = y[train_indices]
y_val = y[val_indices]
y_test = y[test_indices]

y_train_cat = y_categorical[train_indices]
y_val_cat = y_categorical[val_indices]
y_test_cat = y_categorical[test_indices]

print("Text feature dimension:", text_features.shape[1])
print("Audio feature dimension:", audio_features.shape[1])
print("Video feature dimension:", video_features.shape[1])
print("Fused feature dimension:", fused_features_scaled.shape[1])
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)


## 16. Model Architecture

The classifier is a dense neural network:

1. Input layer.
2. Dense layer with ReLU.
3. Dropout.
4. Dense layer with ReLU.
5. Dropout.
6. Softmax output layer.

The TensorFlow version uses Adam optimizer and categorical cross-entropy loss. If TensorFlow is unavailable, a small NumPy neural network with dense layers, ReLU, softmax, and Adam-style updates is used.


In [ ]:
def build_keras_fusion_classifier(input_dim, num_classes):
    model = Sequential(
        [
            Input(shape=(input_dim,)),
            Dense(128, activation="relu"),
            Dropout(0.30),
            Dense(64, activation="relu"),
            Dropout(0.30),
            Dense(num_classes, activation="softmax"),
        ]
    )
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


class NumpyFusionClassifier:
    def __init__(self, input_dim, num_classes, hidden_dim_1=64, hidden_dim_2=32, learning_rate=0.003, random_state=42):
        self.input_dim = input_dim
        self.num_classes = num_classes
        self.hidden_dim_1 = hidden_dim_1
        self.hidden_dim_2 = hidden_dim_2
        self.learning_rate = learning_rate
        self.rng = np.random.default_rng(random_state)
        self._initialize_parameters()

    def _initialize_parameters(self):
        self.W1 = self.rng.normal(0, np.sqrt(2 / self.input_dim), (self.input_dim, self.hidden_dim_1)).astype(np.float32)
        self.b1 = np.zeros(self.hidden_dim_1, dtype=np.float32)
        self.W2 = self.rng.normal(0, np.sqrt(2 / self.hidden_dim_1), (self.hidden_dim_1, self.hidden_dim_2)).astype(np.float32)
        self.b2 = np.zeros(self.hidden_dim_2, dtype=np.float32)
        self.W3 = self.rng.normal(0, np.sqrt(2 / self.hidden_dim_2), (self.hidden_dim_2, self.num_classes)).astype(np.float32)
        self.b3 = np.zeros(self.num_classes, dtype=np.float32)
        self.parameters = ["W1", "b1", "W2", "b2", "W3", "b3"]
        self.adam_m = {name: np.zeros_like(getattr(self, name)) for name in self.parameters}
        self.adam_v = {name: np.zeros_like(getattr(self, name)) for name in self.parameters}
        self.adam_t = 0

    @staticmethod
    def _relu(x):
        return np.maximum(0, x)

    @staticmethod
    def _softmax(logits):
        logits = logits - np.max(logits, axis=1, keepdims=True)
        exp_values = np.exp(logits)
        return exp_values / np.sum(exp_values, axis=1, keepdims=True)

    def _forward(self, X):
        z1 = X @ self.W1 + self.b1
        a1 = self._relu(z1)
        z2 = a1 @ self.W2 + self.b2
        a2 = self._relu(z2)
        logits = a2 @ self.W3 + self.b3
        probabilities = self._softmax(logits)
        cache = {"X": X, "z1": z1, "a1": a1, "z2": z2, "a2": a2}
        return probabilities, cache

    @staticmethod
    def _loss(y_true, probabilities):
        probabilities = np.clip(probabilities, 1e-9, 1.0)
        return float(-np.mean(np.sum(y_true * np.log(probabilities), axis=1)))

    @staticmethod
    def _accuracy(y_true, probabilities):
        return float(np.mean(np.argmax(y_true, axis=1) == np.argmax(probabilities, axis=1)))

    def _apply_adam(self, gradients, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.adam_t += 1
        for name, gradient in gradients.items():
            self.adam_m[name] = beta1 * self.adam_m[name] + (1 - beta1) * gradient
            self.adam_v[name] = beta2 * self.adam_v[name] + (1 - beta2) * (gradient * gradient)
            m_hat = self.adam_m[name] / (1 - beta1 ** self.adam_t)
            v_hat = self.adam_v[name] / (1 - beta2 ** self.adam_t)
            updated = getattr(self, name) - self.learning_rate * m_hat / (np.sqrt(v_hat) + epsilon)
            setattr(self, name, updated.astype(np.float32))

    def fit(self, X, y_cat, validation_data=None, epochs=25, batch_size=128, verbose=1):
        history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
        n_samples = X.shape[0]
        best_val_accuracy = -1.0
        best_epoch = None
        best_state = None

        for epoch in range(1, epochs + 1):
            indices = self.rng.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y_cat[indices]

            for start in range(0, n_samples, batch_size):
                X_batch = X_shuffled[start : start + batch_size]
                y_batch = y_shuffled[start : start + batch_size]
                actual_batch_size = X_batch.shape[0]

                probabilities, cache = self._forward(X_batch)
                dlogits = (probabilities - y_batch) / actual_batch_size

                dW3 = cache["a2"].T @ dlogits
                db3 = np.sum(dlogits, axis=0)
                da2 = dlogits @ self.W3.T
                dz2 = da2 * (cache["z2"] > 0)
                dW2 = cache["a1"].T @ dz2
                db2 = np.sum(dz2, axis=0)
                da1 = dz2 @ self.W2.T
                dz1 = da1 * (cache["z1"] > 0)
                dW1 = cache["X"].T @ dz1
                db1 = np.sum(dz1, axis=0)

                gradients = {
                    "W1": dW1.astype(np.float32),
                    "b1": db1.astype(np.float32),
                    "W2": dW2.astype(np.float32),
                    "b2": db2.astype(np.float32),
                    "W3": dW3.astype(np.float32),
                    "b3": db3.astype(np.float32),
                }
                self._apply_adam(gradients)

            train_probabilities, _ = self._forward(X)
            train_loss = self._loss(y_cat, train_probabilities)
            train_accuracy = self._accuracy(y_cat, train_probabilities)
            history["loss"].append(train_loss)
            history["accuracy"].append(train_accuracy)

            if validation_data is not None:
                X_val_local, y_val_local = validation_data
                val_probabilities, _ = self._forward(X_val_local)
                val_loss = self._loss(y_val_local, val_probabilities)
                val_accuracy = self._accuracy(y_val_local, val_probabilities)
                history["val_loss"].append(val_loss)
                history["val_accuracy"].append(val_accuracy)

                if val_accuracy > best_val_accuracy:
                    best_val_accuracy = val_accuracy
                    best_epoch = epoch
                    best_state = {name: getattr(self, name).copy() for name in self.parameters}

                if verbose and (epoch == 1 or epoch % 5 == 0 or epoch == epochs):
                    print(
                        f"Epoch {epoch:02d}/{epochs} - "
                        f"loss: {train_loss:.4f} - accuracy: {train_accuracy:.4f} - "
                        f"val_loss: {val_loss:.4f} - val_accuracy: {val_accuracy:.4f}"
                    )

        if best_state is not None:
            for name, value in best_state.items():
                setattr(self, name, value)
            if verbose:
                print(f"Restored best NumPy model from epoch {best_epoch} with validation accuracy {best_val_accuracy:.4f}.")

        self.history_ = history
        return history

    def predict_proba(self, X):
        probabilities, _ = self._forward(X)
        return probabilities

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)


input_dim = X_train.shape[1]

if TF_AVAILABLE:
    model = build_keras_fusion_classifier(input_dim, num_classes)
    model.summary()
else:
    model = NumpyFusionClassifier(input_dim, num_classes, random_state=RANDOM_STATE)
    print("Using NumPy dense neural network fallback.")
    print("Input dimension:", input_dim)
    print("Classes:", class_names)


## 17. Model Training

The model is trained on the fused multimodal features.

For TensorFlow/Keras:

- Optimizer: Adam
- Loss: categorical cross-entropy
- Metric: accuracy

For the NumPy fallback, the same core neural network ideas are implemented manually.


In [ ]:
EPOCHS = 25
BATCH_SIZE = 128

if TF_AVAILABLE:
    history_obj = model.fit(
        X_train,
        y_train_cat,
        validation_data=(X_val, y_val_cat),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1,
    )
    history = history_obj.history
else:
    history = model.fit(
        X_train,
        y_train_cat,
        validation_data=(X_val, y_val_cat),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1,
    )

print("Training completed.")


## 18. Evaluation

The model is evaluated using:

- Accuracy
- Precision
- Recall
- F1-score
- Classification report
- Confusion matrix


In [ ]:
def get_prediction_probabilities(model, X):
    if TF_AVAILABLE:
        return model.predict(X, verbose=0)
    return model.predict_proba(X)


def fallback_confusion_matrix(y_true, y_pred, labels):
    matrix = np.zeros((len(labels), len(labels)), dtype=int)
    label_to_position = {label: position for position, label in enumerate(labels)}
    for true_label, predicted_label in zip(y_true, y_pred):
        matrix[label_to_position[true_label], label_to_position[predicted_label]] += 1
    return matrix


def fallback_precision_recall_f1(y_true, y_pred, labels):
    rows = []
    total_support = len(y_true)
    weighted_precision = 0.0
    weighted_recall = 0.0
    weighted_f1 = 0.0

    for label in labels:
        tp = int(np.sum((y_true == label) & (y_pred == label)))
        fp = int(np.sum((y_true != label) & (y_pred == label)))
        fn = int(np.sum((y_true == label) & (y_pred != label)))
        support = int(np.sum(y_true == label))
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append((label, precision, recall, f1, support))
        weighted_precision += precision * support
        weighted_recall += recall * support
        weighted_f1 += f1 * support

    weighted_precision /= max(total_support, 1)
    weighted_recall /= max(total_support, 1)
    weighted_f1 /= max(total_support, 1)
    return rows, weighted_precision, weighted_recall, weighted_f1


def print_fallback_classification_report(y_true, y_pred, class_names):
    labels = np.arange(len(class_names))
    rows, weighted_precision, weighted_recall, weighted_f1 = fallback_precision_recall_f1(y_true, y_pred, labels)

    print(f"{'class':<12} {'precision':>10} {'recall':>10} {'f1-score':>10} {'support':>10}")
    for label, precision, recall, f1, support in rows:
        print(f"{class_names[label]:<12} {precision:>10.4f} {recall:>10.4f} {f1:>10.4f} {support:>10}")
    print(f"{'weighted avg':<12} {weighted_precision:>10.4f} {weighted_recall:>10.4f} {weighted_f1:>10.4f} {len(y_true):>10}")


y_pred_proba = get_prediction_probabilities(model, X_test)
y_pred = np.argmax(y_pred_proba, axis=1)

test_accuracy = float(np.mean(y_pred == y_test))

if SKLEARN_AVAILABLE:
    precision, recall, f1, _ = sklearn_prfs(y_test, y_pred, average="weighted", zero_division=0)
    report = sklearn_classification_report(
        y_test,
        y_pred,
        labels=np.arange(num_classes),
        target_names=class_names,
        zero_division=0,
    )
    cm = sklearn_confusion_matrix(y_test, y_pred, labels=np.arange(num_classes))
else:
    _, precision, recall, f1 = fallback_precision_recall_f1(y_test, y_pred, np.arange(num_classes))
    report = None
    cm = fallback_confusion_matrix(y_test, y_pred, labels=np.arange(num_classes))

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Weighted Precision: {precision:.4f}")
print(f"Weighted Recall: {recall:.4f}")
print(f"Weighted F1-score: {f1:.4f}")

print("\nClassification Report:")
if report is not None:
    print(report)
else:
    print_fallback_classification_report(y_test, y_pred, class_names)

cm_table = pd.DataFrame(cm, index=class_names, columns=class_names)
print("\nConfusion Matrix:")
display(cm_table)

if MATPLOTLIB_AVAILABLE:
    plt.figure(figsize=(8, 6))
    if SEABORN_AVAILABLE:
        sns.heatmap(cm_table, annot=True, fmt="d", cmap="Blues")
    else:
        plt.imshow(cm, cmap="Blues")
        plt.colorbar()
        plt.xticks(np.arange(num_classes), class_names, rotation=45, ha="right")
        plt.yticks(np.arange(num_classes), class_names)
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.show()


## 19. Training Curves

Training curves help check underfitting and overfitting.


In [ ]:
def plot_training_curves(history):
    if not MATPLOTLIB_AVAILABLE:
        print("Matplotlib is unavailable. Showing final metrics instead.")
        print("Last training accuracy:", round(history["accuracy"][-1], 4))
        if history.get("val_accuracy"):
            best_epoch = int(np.argmax(history["val_accuracy"])) + 1
            best_val_accuracy = float(np.max(history["val_accuracy"]))
            print("Best validation epoch:", best_epoch)
            print("Best validation accuracy:", round(best_val_accuracy, 4))
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.get("accuracy", []), label="Training Accuracy")
    axes[0].plot(history.get("val_accuracy", []), label="Validation Accuracy")
    axes[0].set_title("Training vs Validation Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(history.get("loss", []), label="Training Loss")
    axes[1].plot(history.get("val_loss", []), label="Validation Loss")
    axes[1].set_title("Training vs Validation Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


plot_training_curves(history)


## 20. Prediction Function

This function takes:

- text
- audio path
- video path

For MELD, the same `.mp4` path can be passed as both audio and video path because the clip contains audio and visual information.


In [ ]:
def predict_sentiment(text, audio_path="", video_path="", show_output=True):
    text_vector = transform_text_inputs([text])
    audio_vector, audio_used = extract_audio_features(audio_path)
    video_vector, video_used = extract_video_features(video_path)

    fused_vector = np.concatenate(
        [
            text_vector,
            audio_vector.reshape(1, -1),
            video_vector.reshape(1, -1),
        ],
        axis=1,
    ).astype(np.float32)

    fused_vector_scaled = scaler.transform(fused_vector)
    probabilities = get_prediction_probabilities(model, fused_vector_scaled)[0]

    predicted_index = int(np.argmax(probabilities))
    predicted_label = label_encoder.inverse_transform([predicted_index])[0]
    confidence = float(probabilities[predicted_index])

    if show_output:
        print("Input text:", text)
        print("Audio file used:", audio_used)
        print("Video file used:", video_used)
        print("Predicted label:", predicted_label)
        print(f"Confidence score: {confidence:.4f}")
        probability_table = pd.DataFrame(
            {"label": class_names, "probability": probabilities}
        ).sort_values("probability", ascending=False)
        display(probability_table)

    return predicted_label, confidence, probabilities


example_row = df.iloc[0]
print("Example prediction from dataset row")
print("Actual label:", example_row["label"])
predict_sentiment(example_row["text"], example_row["audio_path"], example_row["video_path"])


## 21. Predictions on MELD Dataset Samples

This section predicts a few real dataset samples and compares actual vs predicted labels.


In [ ]:
sample_rows = (
    df.groupby("label", group_keys=False)
    .apply(lambda group: group.sample(n=min(2, len(group)), random_state=RANDOM_STATE))
    .reset_index(drop=True)
)

prediction_rows = []

for _, row in sample_rows.iterrows():
    predicted_label, confidence, _ = predict_sentiment(
        row["text"],
        row["audio_path"],
        row["video_path"],
        show_output=False,
    )
    prediction_rows.append(
        {
            "text": row["text"][:120],
            "actual_label": row["label"],
            "predicted_label": predicted_label,
            "confidence": round(confidence, 4),
            "media_file": Path(str(row["video_path"])).name,
        }
    )

display(pd.DataFrame(prediction_rows))


## 22. What This Notebook Achieved

This notebook now supports real multimodal analysis with MELD:

1. Loads MELD train/dev/test CSV files.
2. Maps each utterance to its matching `.mp4` clip.
3. Uses utterance text for TF-IDF text features.
4. Uses `.mp4` audio for MFCC features when `librosa`/FFmpeg are available.
5. Uses `.mp4` video frames for visual features when OpenCV is available.
6. Concatenates text, audio, and video features using early fusion.
7. Trains a dense neural network classifier.
8. Evaluates using accuracy, precision, recall, F1-score, and confusion matrix.
9. Predicts sentiment/emotion for new samples.

If audio/video dependencies are not installed, the notebook still runs but reports that real audio/video extraction was skipped.


## Conclusion

The project is now a proper multimodal sentiment analysis pipeline when MELD is available. It can use text, audio, and video together.

Future improvements:

1. Use BERT instead of TF-IDF for stronger text embeddings.
2. Use pretrained CNNs such as ResNet, VGG16, or MobileNet for video.
3. Use wav2vec2 or openSMILE for stronger audio features.
4. Use attention-based fusion instead of simple concatenation.
5. Deploy the model using Streamlit or FastAPI.
